# Experimento — Mitra (Amazon)

Placeholder do experimento. O texto abaixo é o roteiro. Não há treino neste arquivo.

Modelo de fundação tabular da Amazon, pré-treinado só em dados sintéticos e usado pelo AutoGluon. O alvo deste notebook é o Mitra v2: transformer de 12 camadas com atenção em linhas e em colunas, cerca de 76 milhões de parâmetros, nos pesos `autogluon/mitra-classifier-2`. O v1 fica como reserva, num bloco próprio, só se o v2 não puder ser executado.


## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

O checkpoint é o v2, `autogluon/mitra-classifier-2`. Sem `hf_model`, o AutoGluon carrega o v1. A arquitetura dos dois é a mesma e está congelada: 12 camadas, largura 512, 4 cabeças, e o feed-forward interno com dimensão 2.048. Cerca de 20 colunas cabem nas 50 features do pré-treino v2 e passam das 16 do v1. Cerca de 7 mil linhas passam do suporte de pré-treino dos dois, 512 no v1 e 5.120 no v2.

Nenhuma fonte diz se a inferência cabe numa GPU de 6 GB. A heurística de reserva do AutoGluon, para este tamanho, dá cerca de 15,5 GB. O arquivo de pesos tem 303 MB, e isso não é o pico de memória. Se faltar memória, o código reduz o suporte.

O default do código é `fine_tune=True`, com 50 passos. O bloco 0 desliga isso.

No modo em contexto, um bloco cada:

- `n_estimators`. Default 1. O bloco sobe o número de cópias cujas probabilidades são promediadas. O artigo usa 32 no ensemble de contexto puro; 32 só entra se os valores menores tiverem vencido, porque trinta e duas cópias pesam na GPU.
- `shuffle_features`. Default `False`. Permuta a ordem das colunas.
- `shuffle_classes`. Default `False`. Permuta a ordem das classes e desfaz na saída.
- `use_random_transforms`. Default `False`. Liga o transformador aleatório do pré-processador, quantil com SVD ou passthrough.
- `random_mirror_x`. Default `True`. Multiplica cada coluna por +1 ou −1. O bloco que desliga só fica se vencer esse default.

O fine-tuning só abre se vencer o contexto. Não há argumento de batch, dropout, weight decay nem de otimizador no dicionário `MITRA`. O otimizador gravado é AdamW. A ativação gravada é GELU. O weight decay de estoque é 0,1. A receita publicada do v2 usa 0,3, e esse número não passa pelo dicionário.

- `fine_tune=True`, sozinho, testa o estoque: 50 passos, `lr` 0,0001, `warmup_steps` 1.000 e `patience` 40. Com esses defaults os 50 passos cabem dentro da rampa, e a taxa nominal quase não chega a valer.
- `warmup_steps`. A receita v2 usa 10. Com a taxa ainda em 0,0001, depois do décimo passo a taxa fica no valor nominal.
- `lr`. A receita v2, em classificação binária com até 16.384 linhas de treino, usa 3e-6. Esse bloco vem depois do warmup curto; com a rampa de 1.000 ainda ligada, essa taxa quase não é alcançada nos 50 passos.
- `patience`. Default 40. Só varia se o early stopping estiver cortando os 50 passos. O checkpoint guarda o menor log loss. Acurácia e AUC não entram aqui: o código trata o número como “menor é melhor”.

`fine_tune_steps` permanece 50. É o default, o valor do artigo e o da receita v2.

O oversampling repetitivo das dobras vale nos dois modos.

Fontes: arXiv:2510.21204, o relatório v2 arXiv:2609.04540, os cards `autogluon/mitra-classifier-2` e `autogluon/mitra-finetune`, e `sklearn_interface.py` do AutoGluon.


## Hiperparâmetros que ainda vão chegar

A pesquisa destes hiperparâmetros está nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. Contexto do v2: `hf_model` em `autogluon/mitra-classifier-2`, `fine_tune` falso, um estimador, espelho de colunas no default, sobre as dobras com oversampling.
2. `n_estimators`, subindo a partir de 1. O 32 do artigo só entra se os valores menores tiverem vencido.
3. `shuffle_features`.
4. `shuffle_classes`.
5. `use_random_transforms`.
6. `random_mirror_x` desligado, contra o default ligado.
7. Portão de fine-tuning: `fine_tune` verdadeiro e mais nada. Só fica se vencer o contexto já aceito.
8. `warmup_steps` em 10, o valor da receita v2.
9. `lr` em 3e-6, a taxa da receita v2 para binário até 16.384 linhas.
10. `patience`, só se o early stopping estiver cortando os 50 passos.
11. Mitra v1, só se o v2 não puder ser executado. Nesse caso o roteiro recomeça do bloco 0 com o checkpoint v1.
12. Estratégias de dados compartilhadas.
13. Bloco final no teste.


## O que este arquivo ainda não contém

Não há código, treino, busca nem número medido. Este notebook é o roteiro. O primeiro bloco só começa depois que a métrica comum estiver escrita.
